# ResumeAgent - 中日英三语智能简历生成器

## 项目简介
解决「一份经历要写三种语言简历」的痛点：Agent 通过主动提问收集求职者信息（用户也可自由补充），
信息沉淀为**唯一的结构化事实库**，再按中、日、英各自的求职规范**独立生成**简历（而非互译），
支持润色、评分与 PDF 导出。

- 中文：单页模块化简历
- 日文：履歴書 ＋ 職務経歴書（双文档、和暦纪年、照片位）
- 英文：ATS 安全版式 Resume

## 作者信息
- 姓名：shiyuanyeming-hub
- GitHub：[@shiyuanyeming-hub](https://github.com/shiyuanyeming-hub)
- 日期：2026-08-13

## 第1部分：环境配置

配置 `.env` 中的 LLM 密钥后即可运行。默认使用 DeepSeek（OpenAI 兼容接口）。

In [ ]:
# ==================================================
# 第1部分：环境配置
# ==================================================

# 安装依赖（如需）
# !pip install -q hello-agents python-dotenv playwright
# !playwright install chromium   # PDF 导出需要浏览器内核（首次运行执行一次）

import itertools
import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter, ToolResponse

# 目录约定：数据放 data/，生成结果放 outputs/
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
FACTS_PATH = OUTPUT_DIR / "user_facts.json"
POLISHED_PATH = OUTPUT_DIR / "polished.json"

print("环境配置完成")

## 第2部分：工具定义

三个确定性工具（不依赖 LLM、结果可复现）：

1. **和暦换算**：西暦 ↔ 令和/平成/昭和/大正/明治，正确处理元号切换边界
2. **事实库管理**：简历结构化数据的读写落盘
3. **三语渲染**：同一份事实库渲染出中文单页 / 日文双文档 / 英文 ATS 版式，每种语言使用各自常见的视觉模板（配色与排版独立设计）

In [ ]:
# ==================================================
# 工具 1：和暦换算（西暦 ↔ 日本年号）
# 按日精度处理元号切换边界（月粒度输入时按该月 1 日计）：
#   令和 2019-05-01 起（2019-04 仍为平成31年4月）
#   平成 1989-01-08 起（1989-01-07 前仍为昭和64年）
#   昭和 1926-12-25 起（1926-12-24 前仍为大正15年）
#   大正 1912-07-30 起（1912-07-29 前仍为明治45年）
#   明治 1868-10-23 起
# ==================================================

ERA_RULES = [
    ("令和", (2019, 5, 1)),
    ("平成", (1989, 1, 8)),
    ("昭和", (1926, 12, 25)),
    ("大正", (1912, 7, 30)),
    ("明治", (1868, 10, 23)),
]

ERA_START_YEAR = {"令和": 2019, "平成": 1989, "昭和": 1926, "大正": 1912, "明治": 1868}

def to_wareki(year: int, month: int = 1, day: int = 1) -> str:
    """西暦 (year, month, day) → 和暦字符串，如 令和5年9月1日"""
    if (year, month, day) < (1868, 10, 23):
        return f"{year}年{month}月{day}日（西暦）"
    for era, (sy, sm, sd) in ERA_RULES:
        if (year, month, day) >= (sy, sm, sd):
            ey = year - ERA_START_YEAR[era] + 1
            return f"{era}{ey}年{month}月{day}日" if ey > 1 else f"{era}元年{month}月{day}日"
    return f"{year}年{month}月{day}日（西暦）"

def wareki_year(era: str, ey: int) -> int:
    """元号 + 年数 → 西暦年"""
    return ERA_START_YEAR[era] + ey - 1

def from_wareki(text: str):
    """和暦字符串 → (year, month, day)，如 '令和5年9月' → (2023, 9, None)"""
    m = re.match(r"(令和|平成|昭和|大正|明治)(?:(\d{1,3})|元)年(\d{1,2})月(?:(\d{1,2})日)?", text)
    if not m:
        raise ValueError(f"无法解析和暦：{text}")
    era, ey_s, mon, day = m.group(1), m.group(2), int(m.group(3)), m.group(4)
    ey = 1 if ey_s is None else int(ey_s)
    return wareki_year(era, ey), mon, (int(day) if day else None)

def to_wareki_date(iso: str) -> str:
    """'2023-09-01' → '令和5年9月1日'；月粒度 '2023-09' → '令和5年9月'"""
    parts = [int(x) for x in iso.split("-")]
    y, m = parts[0], parts[1]
    has_day = len(parts) > 2
    d = parts[2] if has_day else 1
    s = to_wareki(y, m, d)
    return s if has_day else s.replace("1日", "")

print("和暦换算函数就绪")

In [ ]:
# ==================================================
# 工具 1 封装：EraConverterTool（可被 Agent 调用）
# ==================================================

class EraConverterTool(Tool):
    """西暦 ↔ 日本和暦（令和/平成/昭和/大正/明治）互转"""

    def __init__(self):
        super().__init__(
            name="era_converter",
            description="西暦与日本和暦互转。输入如 '2023-09-01 转和暦' 或 '令和5年9月1日 转西暦'。",
        )

    def get_parameters(self):
        return [ToolParameter(name="query", type="string",
                              description="换算请求，含日期字符串与方向", required=True)]

    def run(self, parameters):
        q = parameters.get("query", "")
        iso_m = re.search(r"\d{4}-\d{2}(?:-\d{2})?", q)
        wm = re.search(r"(令和|平成|昭和|大正|明治)(?:\d{1,3}|元)年\d{1,2}月(?:\d{1,2}日)?", q)
        try:
            if iso_m:
                return ToolResponse.success(text=f"{iso_m.group(0)} → {to_wareki_date(iso_m.group(0))}")
            if wm:
                y, m, d = from_wareki(wm.group(0))
                s = f"{wm.group(0)} → {y}年{m}月" + (f"{d}日" if d else "")
                return ToolResponse.success(text=s)
            return ToolResponse.error(code="NO_DATE", message="未识别到日期，请输入如 '2023-09-01 转和暦'")
        except Exception as e:
            return ToolResponse.error(code="CONVERT_FAILED", message=f"换算失败：{e}")

print("EraConverterTool 就绪")

In [ ]:
# ==================================================
# 工具 2：简历结构化事实库（唯一数据源）
# 三语简历均由此渲染；更新会落盘到 outputs/user_facts.json
# ==================================================

DEFAULT_FACTS = {
    "profile": {},
    "education": [],
    "experiences": [],
    "projects": [],
    "skills": {"languages": [], "certifications": [], "tech": []},
    "japan_extra": {},
    "targets": {"role": "", "languages": ["zh", "ja", "en"], "jd": ""},
}

def load_facts() -> dict:
    if FACTS_PATH.exists():
        return json.loads(FACTS_PATH.read_text(encoding="utf-8"))
    return json.loads(json.dumps(DEFAULT_FACTS))

def save_facts(facts: dict) -> None:
    FACTS_PATH.write_text(json.dumps(facts, ensure_ascii=False, indent=2), encoding="utf-8")

class FactsStoreTool(Tool):
    """简历结构化事实库：查询 / JSON 合并更新"""

    def __init__(self):
        super().__init__(
            name="facts_store",
            description="管理简历结构化事实库。query 格式：'查看全部'，或 '更新 {...JSON...}'（按字段合并）。",
        )

    def get_parameters(self):
        return [ToolParameter(name="query", type="string", description="操作指令", required=True)]

    def run(self, parameters):
        q = parameters.get("query", "").strip()
        facts = load_facts()
        if q.startswith("查看") or q.startswith("get"):
            return ToolResponse.success(text=json.dumps(facts, ensure_ascii=False))
        m = re.search(r"\{[\s\S]*\}", q)
        if not m:
            return ToolResponse.error(code="NO_JSON", message="未识别更新内容，请提供 JSON")
        try:
            patch = json.loads(m.group(0))
            for k, v in patch.items():
                if k in facts:
                    facts[k] = v
            save_facts(facts)
            return ToolResponse.success(
                text=f"已更新字段：{list(patch.keys())}，当前共 {len(facts.get('experiences', []))} 段经历"
            )
        except Exception as e:
            return ToolResponse.error(code="UPDATE_FAILED", message=f"更新失败：{e}")

print("FactsStoreTool 就绪")

In [ ]:
# ==================================================
# 工具 3：三语简历渲染器（确定性模板）
# 同一份事实库 → 中文单页 / 日文双文档 / 英文 ATS 安全版式
# ==================================================

def _d(iso: str) -> str:
    """'2023-09' → '2023.09'"""
    return iso.replace("-", ".") if iso else ""

def _ad_jp(iso: str) -> str:
    """'2002-03-15' → '2002年3月15日'"""
    if not iso:
        return ""
    parts = iso.split("-")
    if len(parts) == 3:
        return f"{int(parts[0])}年{int(parts[1])}月{int(parts[2])}日"
    return f"{int(parts[0])}年{int(parts[1])}月"

def _wareki_range(start: str, end: str) -> str:
    if not start:
        return ""
    if not end:
        return f"{to_wareki_date(start)} 〜 現在"
    return f"{to_wareki_date(start)} 〜 {to_wareki_date(end)}"

def _apply_bullets(facts: dict, key: str, polished):
    """把润色后的扁平 bullet 列表按经历逐段填回"""
    if not (polished and polished.get(key)):
        return facts
    flat = polished[key]
    idx = 0
    new_exps = []
    for x in facts.get("experiences", []):
        n = len(x.get("bullets", []))
        seg = flat[idx: idx + n]
        nx = dict(x)
        nx["bullets"] = seg if len(seg) == n else x.get("bullets", [])
        idx += n
        new_exps.append(nx)
    out = dict(facts)
    out["experiences"] = new_exps
    return out

def _summary(facts: dict, lang: str, polished) -> str:
    if polished and polished.get(f"summary_{lang}"):
        return polished[f"summary_{lang}"]
    # 确定性兜底（未运行 LLM 润色时也可渲染）
    if lang == "en":
        return ("Data analyst with hands-on experience in user analytics, "
                "A/B testing, and reporting automation.")
    if lang == "ja":
        return ("データ分析分野で実務経験を積み、ユーザー分析・A/Bテスト・"
                "レポート自動化に取り組んでまいりました。")
    return "具备数据分析实战经验，熟悉用户分析、A/B 实验与报表自动化。"

# ---------- 中文：单页模块化简历 ----------
def render_cn_md(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_zh", polished)
    p = facts.get("profile", {})
    role = facts.get("targets", {}).get("role", "")
    lines = [f"# {p.get('name', '')}", "",
             f"- 电话：{p.get('phone', '')} ｜ 邮箱：{p.get('email', '')}",
             f"- 求职意向：{role}", "",
             "## 教育背景", ""]
    for e in facts.get("education", []):
        lines.append(f"- **{e.get('school', '')}** ｜ {e.get('major', '')} · {e.get('degree', '')}"
                     f" ｜ {_d(e.get('start', ''))}–{_d(e.get('end', ''))}")
        extra = []
        if e.get("gpa"):
            extra.append(f"GPA {e['gpa']}")
        if e.get("honors"):
            extra.append(e["honors"])
        if extra:
            lines.append(f"  - {' ｜ '.join(extra)}")
    lines += ["", "## 工作经历", ""]
    for x in facts.get("experiences", []):
        lines.append(f"- **{x.get('company', '')}** ｜ {x.get('title', '')}"
                     f" ｜ {_d(x.get('start', ''))}–{_d(x.get('end', ''))}")
        for b in x.get("bullets", []):
            lines.append(f"  - {b}")
    lines += ["", "## 技能", ""]
    sk = facts.get("skills", {})
    if sk.get("tech"):
        lines.append(f"- 技术栈：{'、'.join(sk['tech'])}")
    if sk.get("languages"):
        lines.append("- 语言：" + "、".join(f"{l['name']} {l['level']}" for l in sk["languages"]))
    if sk.get("certifications"):
        lines.append("- 证书：" + "、".join(
            f"{c['name']}（{_d(c.get('date', ''))}）" for c in sk["certifications"]))
    lines += ["", "## 自我评价", "", _summary(facts, "zh", polished)]
    return "\n".join(lines)

# ---------- 日文：履歴書 ----------
def _rirekisho_events(facts: dict):
    """学歴・職歴 事件列表（时间正序，年月日式）"""
    events = []
    for e in facts.get("education", []):
        school = e.get("school_ja") or e.get("school", "")
        major = e.get("major_ja") or e.get("major", "")
        if e.get("start"):
            events.append((e["start"], f"{school} {major}学部 入学"))
        if e.get("end"):
            events.append((e["end"], f"{school} 卒業"))
    for x in facts.get("experiences", []):
        company = x.get("company_ja") or x.get("company", "")
        title = x.get("title_ja") or x.get("title", "")
        if x.get("start"):
            events.append((x["start"], f"{company} 入社（{title}）"))
        if x.get("end"):
            events.append((x["end"], f"{company} 退社"))
    events.sort(key=lambda t: t[0])
    return events

def render_rirekisho_md(facts: dict) -> str:
    p = facts.get("profile", {})
    birth_jp = to_wareki_date(p["birth"]) if p.get("birth") else "（未記入）"
    birth_ad = _ad_jp(p.get("birth", ""))
    lines = ["# 履歴書", "",
             f"- **氏名**：{p.get('name', '')}（{p.get('name_kana', '')}）",
             f"- **生年月日**：{birth_jp}" + (f"（{birth_ad}）" if birth_ad else ""),
             f"- **電話**：{p.get('phone', '')}",
             f"- **メール**：{p.get('email', '')}"]
    if p.get("address"):
        lines.append(f"- **現住所**：{p['address']}")
    if p.get("nearest_station"):
        lines.append(f"- **最寄駅**：{p['nearest_station']}")
    lines.append("- **写真**：（3×4cm 証明写真貼付欄）")
    lines += ["", "## 学歴・職歴", ""]
    for date, text in _rirekisho_events(facts):
        lines.append(f"- {to_wareki_date(date)}　{text}")
    lines += ["", "## 免許・資格", ""]
    certs = facts.get("skills", {}).get("certifications", [])
    if certs:
        for c in certs:
            lines.append(f"- {c.get('name_ja') or c.get('name', '')}"
                         + (f"（{to_wareki_date(c['date'])}）" if c.get("date") else ""))
    else:
        lines.append("- 特になし")
    je = facts.get("japan_extra", {})
    lines += ["", "## 志望動機", "", je.get("motivation", "（未記入）"), "",
              "## 本人希望欄", "", je.get("desired_position", "貴社規定に従います。"), ""]
    return "\n".join(lines)

# ---------- 日文：職務経歴書 ----------
def render_shokumu_md(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_ja", polished)
    p = facts.get("profile", {})
    lines = [f"# 職務経歴書", "", f"氏名：{p.get('name', '')}", "", "## 職務要約", "",
             _summary(facts, "ja", polished), "", "## 職務経歴", ""]
    for x in sorted(facts.get("experiences", []), key=lambda t: t.get("start", ""), reverse=True):
        company = x.get("company_ja") or x.get("company", "")
        title = x.get("title_ja") or x.get("title", "")
        lines.append(f"### {_wareki_range(x.get('start', ''), x.get('end', ''))}　{company}（{title}）")
        lines.append("")
        for b in x.get("bullets", []):
            lines.append(f"- {b}")
        lines.append("")
    lines += ["## 活かせるスキル", ""]
    sk = facts.get("skills", {})
    if sk.get("tech"):
        lines.append("- 技術：" + "、".join(sk["tech"]))
    if sk.get("languages"):
        lines.append("- 語学：" + "、".join(f"{l['name']}（{l['level']}）" for l in sk["languages"]))
    return "\n".join(lines)

# ---------- 英文：ATS 安全版式 ----------
def render_en_md(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_en", polished)
    p = facts.get("profile", {})
    lines = [f"# {p.get('name', '')}", "",
             f"Phone: {p.get('phone', '')}  |  Email: {p.get('email', '')}", "",
             "## Summary", "", _summary(facts, "en", polished), "",
             "## Experience", ""]
    for x in sorted(facts.get("experiences", []), key=lambda t: t.get("start", ""), reverse=True):
        company = x.get("company_en") or x.get("company", "")
        title = x.get("title_en") or x.get("title", "")
        lines.append(f"### {title} — {company} | {_d(x.get('start', ''))} – {_d(x.get('end', ''))}")
        lines.append("")
        for b in x.get("bullets", []):
            lines.append(f"- {b}")
        lines.append("")
    lines += ["## Education", ""]
    for e in facts.get("education", []):
        school = e.get("school_en") or e.get("school", "")
        major = e.get("major_en") or e.get("major", "")
        deg = e.get("degree_en") or e.get("degree", "")
        lines.append(f"### {school} — {deg} in {major} | {_d(e.get('start', ''))} – {_d(e.get('end', ''))}")
        if e.get("gpa"):
            lines.append(f"- GPA: {e['gpa']}" + (f"; {e['honors']}" if e.get("honors") else ""))
        lines.append("")
    lines += ["## Skills", ""]
    sk = facts.get("skills", {})
    if sk.get("tech"):
        lines.append("- **Technical:** " + ", ".join(sk["tech"]))
    if sk.get("languages"):
        lines.append("- **Languages:** " + ", ".join(
            f"{l.get('name_en') or l['name']} ({l['level']})" for l in sk["languages"]))
    if sk.get("certifications"):
        lines.append("- **Certifications:** " + ", ".join(
            f"{c.get('name_en') or c['name']} ({_d(c.get('date', ''))})" for c in sk["certifications"]))
    return "\n".join(lines)

print("三语渲染函数就绪")

In [ ]:
# ==================================================
# HTML 版式：三种语言各走独立视觉模板（A4 打印友好）
#   中文：藏青蓝现代单页（色块标题 + 技能标签）
#   日文：履歴書 JIS 表单（藏青表头/照片位）＋ 職務経歴書
#   英文：ATS 安全单栏 + 青灰强调色（标准标题名不变，不影响机器解析）
# ==================================================

CSS_THEMES = {
    # ---------- 中文：藏青蓝 + 浅蓝底 ----------
    "zh": """
@page { size: A4; margin: 14mm 16mm; }
body { font-family: "PingFang SC", "Hiragino Sans GB", "Noto Sans CJK SC", sans-serif;
       font-size: 10pt; line-height: 1.7; color: #2b2b2b; margin: 0; }
.header { border-bottom: 2.2pt solid #1F4E79; padding-bottom: 3mm; margin-bottom: 4.5mm; }
.name { font-size: 20pt; font-weight: bold; color: #1F4E79; margin: 0; letter-spacing: 1px; }
.role { color: #4A6785; font-size: 10.5pt; margin: 1.5mm 0 0; }
.contact { color: #666; font-size: 9pt; margin-top: 1.5mm; }
.contact span { margin-right: 4mm; }
h2 { font-size: 11.5pt; color: #1F4E79; margin: 4.5mm 0 2mm; padding-left: 2.5mm;
     border-left: 3.5pt solid #1F4E79; }
h3 { font-size: 10.5pt; color: #2b2b2b; margin: 2.5mm 0 0.5mm; }
.muted { color: #666; font-size: 9pt; }
ul { margin: 1mm 0 2mm; padding-left: 5mm; }
li { margin: 0.7mm 0; }
li::marker { color: #4A90B8; }
.chips { margin: 1mm 0 2mm; }
.chip { display: inline-block; background: #EDF2F8; border: 0.5pt solid #B8CCE4;
        border-radius: 2.5mm; padding: 0.4mm 2.2mm; margin: 0.4mm; font-size: 9pt; color: #1F4E79; }
.job-title { color: #1F4E79; font-weight: bold; }
""",
    # ---------- 英文：ATS 安全单栏 + 青灰强调 ----------
    "en": """
@page { size: A4; margin: 14mm 16mm; }
body { font-family: "Helvetica Neue", Arial, "PingFang SC", sans-serif;
       font-size: 10pt; line-height: 1.55; color: #1f2937; margin: 0; }
.name { font-size: 19pt; font-weight: bold; color: #0F4C5C; letter-spacing: 0.5px; margin: 0; }
.contact { color: #555; font-size: 9.5pt; margin: 1mm 0 0; }
h2 { font-size: 11pt; letter-spacing: 0.6px; color: #0F4C5C; text-transform: uppercase;
     border-bottom: 1pt solid #0F4C5C; padding-bottom: 0.8mm; margin: 4.5mm 0 2mm; }
h3 { font-size: 10.5pt; margin: 2.5mm 0 0.5mm; color: #111; }
h3 .dates { font-weight: normal; color: #555; font-size: 9.5pt; }
.muted { color: #555; font-size: 9.5pt; }
ul { margin: 1mm 0 2.5mm; padding-left: 5mm; }
li { margin: 0.7mm 0; }
""",
    # ---------- 日文 履歴書：JIS 表单 + 藏青配色 ----------
    "jp_form": """
@page { size: A4; margin: 12mm; }
body { font-family: "Hiragino Kaku Gothic ProN", "Hiragino Sans GB", "Noto Sans CJK JP", sans-serif;
       font-size: 10.5pt; line-height: 1.6; color: #1a1a1a; margin: 0; }
.title-band { background: #223A5E; color: #fff; text-align: center; font-size: 14pt;
              font-weight: bold; letter-spacing: 6px; padding: 2mm 0; margin-bottom: 3mm; }
.head-row { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 2.5mm; }
.person { font-size: 13pt; font-weight: bold; }
.person .kana { font-size: 10pt; color: #555; font-weight: normal; }
.photo-box { width: 26mm; height: 34mm; border: 0.8pt solid #223A5E; background: #F4F7FA;
             color: #8A9BB0; font-size: 8pt; display: flex; align-items: center;
             justify-content: center; text-align: center; }
table.grid { width: 100%; border-collapse: collapse; }
table.grid td, table.grid th { border: 0.7pt solid #223A5E; padding: 1.6mm 3mm; vertical-align: top; }
.kv td:first-child { width: 26mm; background: #EDF1F6; font-weight: bold; color: #223A5E; }
table.grid th { background: #EDF1F6; color: #223A5E; }
h2 { font-size: 11.5pt; color: #223A5E; border-bottom: 1.2pt solid #223A5E;
     padding-bottom: 0.8mm; margin: 3.8mm 0 1.6mm; }
ul { margin: 1mm 0 2mm; padding-left: 5mm; }
li { margin: 0.7mm 0; }
""",
    # ---------- 日文 職務経歴書 ----------
    "jp_doc": """
@page { size: A4; margin: 15mm; }
body { font-family: "Hiragino Kaku Gothic ProN", "Hiragino Sans GB", "Noto Sans CJK JP", sans-serif;
       font-size: 10.5pt; line-height: 1.7; color: #1a1a1a; margin: 0; }
.doc-title { font-size: 16pt; font-weight: bold; color: #223A5E; text-align: center;
             letter-spacing: 3px; border-bottom: 2pt solid #223A5E; padding-bottom: 2mm; margin-bottom: 3mm; }
.author-line { text-align: right; color: #333; margin-bottom: 3mm; }
h2 { font-size: 11.5pt; color: #223A5E; padding-left: 2.5mm; border-left: 3.5pt solid #223A5E;
     margin: 4.5mm 0 2mm; }
h3 { font-size: 10.5pt; color: #223A5E; margin: 2.5mm 0 0.5mm; }
ul { margin: 1mm 0 2mm; padding-left: 5mm; }
li { margin: 0.7mm 0; }
li::marker { color: #5A7A9E; }
p { margin: 1mm 0; }
""",
}

def _html_page(title: str, body: str, theme: str) -> str:
    return (f'<!DOCTYPE html>\n<html lang="zh"><head><meta charset="utf-8"><title>{title}</title>\n'
            f'<style>{CSS_THEMES[theme]}</style></head>\n<body>{body}</body></html>')

# ---------- 中文 HTML：藏青现代版式 ----------
def _zh_html(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_zh", polished)
    p = facts.get("profile", {})
    role = facts.get("targets", {}).get("role", "")
    head = (
        '<div class="header">'
        f'<p class="name">{p.get("name", "")}</p>'
        f'<p class="role">求职意向：{role}</p>'
        f'<p class="contact"><span>电话：{p.get("phone", "")}</span>'
        f'<span>邮箱：{p.get("email", "")}</span></p></div>'
    )
    edu = ""
    for e in facts.get("education", []):
        extra = []
        if e.get("gpa"):
            extra.append(f"GPA {e['gpa']}")
        if e.get("honors"):
            extra.append(e["honors"])
        edu += (f'<h3>{e.get("school", "")} <span class="muted">{e.get("major", "")} · '
                f'{e.get("degree", "")} ｜ {_d(e.get("start", ""))}–{_d(e.get("end", ""))}</span></h3>')
        if extra:
            edu += f'<p class="muted">{(" ｜ ".join(extra))}</p>'
    exp = ""
    for x in facts.get("experiences", []):
        exp += (f'<h3><span class="job-title">{x.get("company", "")}</span> '
                f'<span class="muted">{x.get("title", "")} ｜ {_d(x.get("start", ""))}–{_d(x.get("end", ""))}</span></h3><ul>'
                + "".join(f"<li>{b}</li>" for b in x.get("bullets", [])) + "</ul>")
    sk = facts.get("skills", {})
    chips = []
    if sk.get("tech"):
        chips += sk["tech"]
    if sk.get("languages"):
        chips += [f"{l['name']} {l['level']}" for l in sk["languages"]]
    chips_html = '<div class="chips">' + "".join(f'<span class="chip">{c}</span>' for c in chips) + "</div>"
    body = (head + "<h2>教育背景</h2>" + edu + "<h2>工作经历</h2>" + exp
            + "<h2>技能</h2>" + chips_html
            + "<h2>自我评价</h2><p>" + _summary(facts, "zh", polished) + "</p>")
    return _html_page("简历", body, "zh")

# ---------- 英文 HTML：ATS 安全 + 青灰强调 ----------
def _en_html(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_en", polished)
    p = facts.get("profile", {})
    head = (f'<p class="name">{p.get("name", "")}</p>'
            f'<p class="contact">Phone: {p.get("phone", "")} | Email: {p.get("email", "")}</p>')
    exp = ""
    for x in sorted(facts.get("experiences", []), key=lambda t: t.get("start", ""), reverse=True):
        company = x.get("company_en") or x.get("company", "")
        title = x.get("title_en") or x.get("title", "")
        exp += (f'<h3>{title} — {company} <span class="dates">{_d(x.get("start", ""))} – {_d(x.get("end", ""))}</span></h3><ul>'
                + "".join(f"<li>{b}</li>" for b in x.get("bullets", [])) + "</ul>")
    edu = ""
    for e in facts.get("education", []):
        school = e.get("school_en") or e.get("school", "")
        major = e.get("major_en") or e.get("major", "")
        deg = e.get("degree_en") or e.get("degree", "")
        edu += (f'<h3>{school} — {deg} in {major} <span class="dates">{_d(e.get("start", ""))} – {_d(e.get("end", ""))}</span></h3>')
        if e.get("gpa"):
            edu += f'<p class="muted">GPA: {e["gpa"]}' + (f"; {e['honors']}" if e.get("honors") else "") + "</p>"
    sk = facts.get("skills", {})
    lis = []
    if sk.get("tech"):
        lis.append(f'<li><b>Technical:</b> {", ".join(sk["tech"])}</li>')
    if sk.get("languages"):
        lis.append('<li><b>Languages:</b> ' + ", ".join(
            f"{l.get('name_en') or l['name']} ({l['level']})" for l in sk["languages"]) + "</li>")
    if sk.get("certifications"):
        lis.append('<li><b>Certifications:</b> ' + ", ".join(
            f"{c.get('name_en') or c['name']} ({_d(c.get('date', ''))})" for c in sk["certifications"]) + "</li>")
    skills = "<ul>" + "".join(lis) + "</ul>"
    body = (head + "<h2>Summary</h2><p>" + _summary(facts, "en", polished) + "</p>"
            + "<h2>Experience</h2>" + exp
            + "<h2>Education</h2>" + edu
            + "<h2>Skills</h2>" + skills)
    return _html_page("Resume", body, "en")

# ---------- 日文 履歴書 HTML：JIS 表单 ----------
def _rirekisho_html(facts: dict) -> str:
    p = facts.get("profile", {})
    birth_jp = to_wareki_date(p["birth"]) if p.get("birth") else "（未記入）"
    birth_ad = _ad_jp(p.get("birth", ""))
    head = (
        '<div class="title-band">履　歴　書</div>'
        '<div class="head-row">'
        f'<div class="person">{p.get("name", "")} <span class="kana">（{p.get("name_kana", "")}）</span></div>'
        '<div class="photo-box">写真<br>（3×4cm）</div></div>'
    )
    kv_rows = [
        ("生年月日", birth_jp + (f"（{birth_ad}）" if birth_ad else "")),
        ("電話", p.get("phone", "")),
        ("メール", p.get("email", "")),
        ("現住所", p.get("address", "")),
        ("最寄駅", p.get("nearest_station", "")),
    ]
    kv = ('<table class="grid kv">'
          + "".join(f"<tr><td>{k}</td><td>{v or '（未記入）'}</td></tr>" for k, v in kv_rows)
          + "</table>")
    ev_rows = "".join(f"<tr><td>{to_wareki_date(d)}</td><td>{t}</td></tr>"
                      for d, t in _rirekisho_events(facts))
    ev = ('<h2>学歴・職歴</h2><table class="grid">'
          '<tr><th style="width:34mm;">年　　月</th><th>学歴・職歴</th></tr>'
          + ev_rows + "</table>")
    certs = facts.get("skills", {}).get("certifications", [])
    cert_html = ("<ul>" + "".join(
        f"<li>{c.get('name_ja') or c.get('name', '')}"
        + (f"（{to_wareki_date(c['date'])}）" if c.get("date") else "") + "</li>"
        for c in certs) + "</ul>") if certs else "<p>特になし</p>"
    je = facts.get("japan_extra", {})
    body = (head + kv + ev + "<h2>免許・資格</h2>" + cert_html
            + "<h2>志望動機</h2><p>" + (je.get("motivation") or "（未記入）") + "</p>"
            + "<h2>本人希望欄</h2><p>" + (je.get("desired_position") or "貴社規定に従います。") + "</p>")
    return _html_page("履歴書", body, "jp_form")

# ---------- 日文 職務経歴書 HTML ----------
def _shokumu_html(facts: dict, polished=None) -> str:
    facts = _apply_bullets(facts, "bullets_ja", polished)
    p = facts.get("profile", {})
    exp = ""
    for x in sorted(facts.get("experiences", []), key=lambda t: t.get("start", ""), reverse=True):
        company = x.get("company_ja") or x.get("company", "")
        title = x.get("title_ja") or x.get("title", "")
        exp += (f'<h3>{_wareki_range(x.get("start", ""), x.get("end", ""))}　{company}（{title}）</h3><ul>'
                + "".join(f"<li>{b}</li>" for b in x.get("bullets", [])) + "</ul>")
    sk = facts.get("skills", {})
    lis = []
    if sk.get("tech"):
        lis.append("<li>技術：" + "、".join(sk["tech"]) + "</li>")
    if sk.get("languages"):
        lis.append("<li>語学：" + "、".join(f"{l['name']}（{l['level']}）" for l in sk["languages"]) + "</li>")
    skills = "<ul>" + "".join(lis) + "</ul>" if lis else "<p>特になし</p>"
    body = ('<p class="doc-title">職務経歴書</p>'
            + f'<p class="author-line">{p.get("name", "")}</p>'
            + "<h2>職務要約</h2><p>" + _summary(facts, "ja", polished) + "</p>"
            + "<h2>職務経歴</h2>" + exp
            + "<h2>活かせるスキル</h2>" + skills)
    return _html_page("職務経歴書", body, "jp_doc")

def export_all(facts: dict, polished=None, out_dir: Path = OUTPUT_DIR) -> dict:
    """渲染并写出三语简历（md + html），每种语言使用独立视觉模板"""
    results = {}

    zh_md = render_cn_md(facts, polished)
    results["zh"] = {"md": out_dir / "resume_zh.md", "html": out_dir / "resume_zh.html"}
    results["zh"]["md"].write_text(zh_md, encoding="utf-8")
    results["zh"]["html"].write_text(_zh_html(facts, polished), encoding="utf-8")

    ri_md = render_rirekisho_md(facts)
    results["rirekisho"] = {"md": out_dir / "rirekisho_ja.md", "html": out_dir / "rirekisho_ja.html"}
    results["rirekisho"]["md"].write_text(ri_md, encoding="utf-8")
    results["rirekisho"]["html"].write_text(_rirekisho_html(facts), encoding="utf-8")

    sh_md = render_shokumu_md(facts, polished)
    results["shokumu"] = {"md": out_dir / "shokumu_keirekisho_ja.md",
                          "html": out_dir / "shokumu_keirekisho_ja.html"}
    results["shokumu"]["md"].write_text(sh_md, encoding="utf-8")
    results["shokumu"]["html"].write_text(_shokumu_html(facts, polished), encoding="utf-8")

    en_md = render_en_md(facts, polished)
    results["en"] = {"md": out_dir / "resume_en.md", "html": out_dir / "resume_en.html"}
    results["en"]["md"].write_text(en_md, encoding="utf-8")
    results["en"]["html"].write_text(_en_html(facts, polished), encoding="utf-8")
    return results

print("HTML 版式与导出函数就绪")

In [ ]:
# ==================================================
# PDF 导出：优先 Playwright Chromium，其次本机 Chrome/Edge，
# 都没有时提示用浏览器打印 HTML（版式已按 A4 排好）
# ==================================================

def html_to_pdf(html_path: Path, pdf_path: Path) -> str:
    uri = html_path.resolve().as_uri()
    # 方案 1：Playwright Chromium
    try:
        from playwright.sync_api import sync_playwright
        with sync_playwright() as pw:
            browser = pw.chromium.launch()
            page = browser.new_page()
            page.goto(uri)
            page.pdf(path=str(pdf_path), format="A4", print_background=True)
            browser.close()
        return "playwright"
    except Exception:
        pass
    # 方案 2：本机 Chrome / Edge headless
    # 说明：--no-sandbox 仅为兼容受限运行环境（如容器/CI），本机正常环境可去掉
    candidates = [
        "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
        "/Applications/Microsoft Edge.app/Contents/MacOS/Microsoft Edge",
        shutil.which("google-chrome"), shutil.which("chromium"), shutil.which("msedge"),
    ]
    for c in candidates:
        if c and Path(c).exists():
            try:
                subprocess.run([c, "--headless=new", "--disable-gpu", "--no-sandbox",
                                "--no-pdf-header-footer",
                                f"--print-to-pdf={pdf_path}", uri],
                               capture_output=True, timeout=120)
                if pdf_path.exists():
                    return "chrome"
            except Exception:
                continue
    print(f"未找到可用的 PDF 内核，请用浏览器打开 {html_path.name} 后 Ctrl/Cmd+P 另存为 PDF")
    return "browser-print"

print("PDF 导出函数就绪")

## 第3部分：智能体构建

四个智能体分工：

| 智能体 | 职责 |
|--------|------|
| 访谈官-提问 | 按信息缺口提出下一轮问题（模块化推进 + 量化追问） |
| 访谈官-解析 | 把自然语言回答解析为结构化事实（禁止编造） |
| 润色 ×3（中/日/英） | 按各语言求职规范改写要点（动词开头 / ですます体 / ATS 关键词） |
| 简历评审 | 模拟 HR + ATS 视角评分并给出修改建议 |

In [ ]:
# ==================================================
# LLM 客户端（OpenAI 兼容接口，默认 DeepSeek）
# ==================================================

def get_llm(temperature: float = 0.5, max_tokens: int = 2048) -> HelloAgentsLLM:
    return HelloAgentsLLM(
        model=os.getenv("LLM_MODEL_ID", "deepseek-chat"),
        api_key=os.getenv("LLM_API_KEY") or os.getenv("DEEPSEEK_API_KEY", ""),
        base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"),
        temperature=temperature,
        max_tokens=max_tokens,
        timeout=int(os.getenv("LLM_TIMEOUT", "60")),
    )

try:
    llm = get_llm()
    print(f"LLM 就绪：{llm.model}")
except Exception as e:
    llm = None
    print(f"LLM 未配置（{e}）")
    print("提示：和暦换算、三语渲染等本地功能仍可运行；"
          "Agent 功能需在 .env 中配置 LLM_API_KEY / LLM_BASE_URL")

In [ ]:
# ==================================================
# 智能体提示词与构建
# ==================================================

INTERVIEW_ASK_PROMPT = """你是资深求职顾问，通过面试式提问收集求职者信息，用于生成中日英三语简历。
规则：
1. 每次只问 1~2 个问题，聚焦当前信息缺口，不重复已收集的信息；
2. 按模块顺序推进：基本信息 → 教育背景 → 工作经历 → 项目/技能 → 日本求职补充信息（志望動機・自己PR，仅当求职目标包含日文时需要）；
3. 主动追问量化细节（规模、增幅、效率、预算、团队人数）；
4. 只输出 JSON：{"module": "模块名", "questions": ["问题1", "问题2"]}，不要输出任何其他内容。"""

INTERVIEW_PARSE_PROMPT = """你负责把求职者的自然语言回答解析为结构化事实，写入简历事实库。
事实库结构（JSON）：profile / education / experiences / projects / skills / japan_extra。
规则：
1. 只抽取用户明确说出的信息，禁止编造；
2. experiences 输出为完整数组（保留已有条目，补全/更新新条目）；
3. 每条经历中的 bullets 用简洁、量化的中文短句；
4. 只输出 JSON：{"updates": {字段: 值}}，不要输出任何其他内容。"""

POLISH_PROMPTS = {
    "zh": ("你是中文简历写作专家。把给出的经历要点改写为专业、简洁的中文简历条目。"
           "规则：1. 动词开头；2. 保留全部数字与事实，不得编造；3. 每条不超过 40 字，突出成果；"
           '4. 只输出 JSON：{"bullets": ["..."]}。'),
    "en": ("You are an expert English resume writer. Rewrite the given experience bullets into "
           "ATS-friendly English resume bullets. Rules: 1. Start each bullet with a strong "
           "past-tense action verb (e.g., Led, Built, Analyzed, Automated); 2. Keep ALL numbers "
           "and facts unchanged, never invent data; 3. Each bullet no longer than 2 lines, "
           'highlight results (STAR); 4. Output ONLY JSON: {"bullets": ["..."]}.'),
    "ja": ("あなたは職務経歴書作成の専門家です。与えられた実績を簡潔でプロフェッショナルな"
           "日本語（です・ます体）に書き直してください。ルール：1. 数字と事実はすべて保持し、"
           "捏造しない；2. 各項目は1〜2行以内；3. 成果を強調；"
           '4. JSONのみ出力：{"bullets": ["..."]}。'),
}

SUMMARY_PROMPTS = {
    "zh": "基于给出的简历事实，写一段 60 字以内的中文自我评价。只基于事实，不编造。"
          '只输出 JSON：{"summary": "..."}。',
    "en": ("Write a 2-3 sentence professional summary for an English resume based on the given "
           'facts only. Do not invent anything. Output ONLY JSON: {"summary": "..."}.'),
    "ja": ("与えられた事実のみに基づき、職務経歴書の「職務要約」を日本語で2〜3文作成してください。"
           '捏造禁止。JSONのみ出力：{"summary": "..."}。'),
}

REVIEW_PROMPT = """你是资深 HR，同时熟悉 ATS 系统。请审阅简历（英文）并输出 JSON：
{"scores": {"ats": 0-10, "quantification": 0-10, "structure": 0-10, "language": 0-10},
 "issues": ["问题1", "问题2"],
 "suggestions": ["建议1", "建议2"]}
评分标准：ats=ATS解析友好度；quantification=量化程度；structure=结构清晰度；language=语言专业度。"""

def _extract_json(text: str) -> dict:
    """从 LLM 输出中容错提取 JSON"""
    text = (text or "").strip()
    m = re.search(r"\{[\s\S]*\}", text)
    return json.loads(m.group(0)) if m else {}

def build_all_agents(llm):
    ask_agent = SimpleAgent(name="访谈官-提问", llm=llm, system_prompt=INTERVIEW_ASK_PROMPT)
    parse_agent = SimpleAgent(name="访谈官-解析", llm=llm, system_prompt=INTERVIEW_PARSE_PROMPT)
    polish_agents = {lang: SimpleAgent(name=f"润色-{lang}", llm=llm, system_prompt=p)
                     for lang, p in POLISH_PROMPTS.items()}
    summary_agents = {lang: SimpleAgent(name=f"摘要-{lang}", llm=llm, system_prompt=p)
                      for lang, p in SUMMARY_PROMPTS.items()}
    review_agent = SimpleAgent(name="简历评审", llm=llm, system_prompt=REVIEW_PROMPT)
    return ask_agent, parse_agent, polish_agents, summary_agents, review_agent

if llm is not None:
    ask_agent, parse_agent, polish_agents, summary_agents, review_agent = build_all_agents(llm)
    print("智能体构建完成：访谈官×2、润色×3、摘要×3、评审×1")
else:
    print("跳过智能体构建（LLM 未配置）")

## 第4部分：功能演示

演示流程：工具直调 → Agent 访谈 → 三语生成 → 润色对比 → 模拟评审 → PDF 导出

In [ ]:
# ==================================================
# 演示 1：和暦换算工具（确定性计算，不消耗 LLM）
# ==================================================

registry = ToolRegistry()
era_tool = EraConverterTool()
facts_tool = FactsStoreTool()
registry.register_tool(era_tool)
registry.register_tool(facts_tool)

print("=== 演示 1：和暦换算 ===")
for q in ["2023-09-01 转和暦", "令和5年9月1日 转西暦", "1989-01-10 转和暦"]:
    resp = registry.execute_tool("era_converter", q)
    print(f"[{resp.status.value}] {q} → {resp.text}")

In [ ]:
# ==================================================
# 演示 2：Agent 访谈（提问 → 用户回答 → 解析入库）
# ==================================================

sample = json.loads((DATA_DIR / "sample_answers.json").read_text(encoding="utf-8"))
facts = {k: v for k, v in sample.items() if k in DEFAULT_FACTS}
# 制造信息缺口：第二段经历暂无要点
facts["experiences"][1]["bullets"] = []
save_facts(facts)

print("=== 演示 2：Agent 访谈 ===")
print("\n【访谈前】第二段经历（存在信息缺口）：")
print(json.dumps(facts["experiences"][1], ensure_ascii=False, indent=2))

q_resp = ask_agent.run("当前事实库：" + json.dumps(facts, ensure_ascii=False)
                       + "\n请针对信息缺口提出下一轮问题")
print("\n【Agent 提问】")
print(q_resp)

answer = sample["demo_answer"]
print(f"\n【用户回答】{answer}")

parse_resp = parse_agent.run(f"用户回答：{answer}\n当前事实库：{json.dumps(facts, ensure_ascii=False)}"
                             "\n请解析并输出 updates JSON")
print("\n【Agent 解析】")
print(parse_resp)

updates = _extract_json(parse_resp).get("updates", {})
for k, v in updates.items():
    if k in facts:
        facts[k] = v
save_facts(facts)

print("\n【更新后】第二段经历要点：")
print(json.dumps(facts["experiences"][1]["bullets"], ensure_ascii=False, indent=2))

In [ ]:
# ==================================================
# 演示 3：三语简历生成（同一事实库 → 四份文档）
# ==================================================

print("=== 演示 3：三语生成 ===")
files = export_all(facts)
for lang, paths in files.items():
    for kind, p in paths.items():
        size = p.stat().st_size
        print(f"{lang:>9} {kind:>4}: {p.name}（{size} B）")

print("\n--- 履歴書 预览（前 26 行）---")
print("\n".join(files["rirekisho"]["md"].read_text(encoding="utf-8").split("\n")[:26]))

print("\n--- 英文 Resume 预览（前 16 行）---")
print("\n".join(files["en"]["md"].read_text(encoding="utf-8").split("\n")[:16]))

In [ ]:
# ==================================================
# 演示 4：三语润色 + 摘要（LLM）
# ==================================================

print("=== 演示 4：润色对比 ===")
raw_bullets = [b for x in facts["experiences"] for b in x.get("bullets", [])]
polished = {}

for lang in ["zh", "ja", "en"]:
    t0 = time.perf_counter()
    resp = polish_agents[lang].run("要点：" + json.dumps(raw_bullets, ensure_ascii=False))
    dt = time.perf_counter() - t0
    data = _extract_json(resp)
    polished[f"bullets_{lang}"] = data.get("bullets", raw_bullets)
    print(f"[{lang}] 润色完成，耗时 {dt:.1f}s，共 {len(polished[f'bullets_{lang}'])} 条")

for o, n in itertools.zip_longest(raw_bullets, polished.get("bullets_en", [])):
    if o is not None:
        print(f"\n原文：{o}")
    if n is not None:
        print(f"英文：{n}")

for lang in ["zh", "ja", "en"]:
    resp = summary_agents[lang].run("事实：" + json.dumps(facts, ensure_ascii=False))
    polished[f"summary_{lang}"] = _extract_json(resp).get("summary", "")
    print(f"\n[{lang}] 摘要：{polished[f'summary_{lang}']}")

POLISHED_PATH.write_text(json.dumps(polished, ensure_ascii=False, indent=2), encoding="utf-8")

# 用润色结果重新渲染
files = export_all(facts, polished)
print("\n已用润色结果重新渲染三语简历")

In [ ]:
# ==================================================
# 演示 5：模拟 HR / ATS 评审
# ==================================================

print("=== 演示 5：简历评审 ===")
en_md = files["en"]["md"].read_text(encoding="utf-8")
resp = review_agent.run("简历：\n" + en_md + "\n目标岗位：" + facts["targets"]["role"])
review = _extract_json(resp)

for k, v in review.get("scores", {}).items():
    filled = int(v / 2)
    bar = "█" * filled + "░" * (5 - filled)
    print(f"{k:>14}: {bar} {v}/10")
for i, iss in enumerate(review.get("issues", []), 1):
    print(f"问题 {i}：{iss}")
for i, sug in enumerate(review.get("suggestions", []), 1):
    print(f"建议 {i}：{sug}")

In [ ]:
# ==================================================
# 演示 6：PDF 导出（A4，CJK 字体由系统字体保证）
# ==================================================

print("=== 演示 6：PDF 导出 ===")
pdf_targets = [
    ("resume_zh.html", "resume_zh.pdf"),
    ("rirekisho_ja.html", "rirekisho_ja.pdf"),
    ("shokumu_keirekisho_ja.html", "shokumu_keirekisho_ja.pdf"),
    ("resume_en.html", "resume_en.pdf"),
]
for h, p in pdf_targets:
    method = html_to_pdf(OUTPUT_DIR / h, OUTPUT_DIR / p)
    print(f"{h} → {p}（内核：{method}）")

print("\n全部产物已写入 outputs/ 目录：")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {p.name}（{p.stat().st_size} B）")

In [ ]:
# ==================================================
# 交互模式：把 RUN_INTERACTIVE 改为 True 即可与 Agent 实时对话
# ==================================================

RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    facts = load_facts()
    for i in range(3):
        q = ask_agent.run("当前事实库：" + json.dumps(facts, ensure_ascii=False)
                          + "\n请提出下一轮问题")
        print(f"\n【第{i+1}轮】{q}")
        ans = input("你的回答（直接回车跳过）：").strip()
        if not ans:
            continue
        resp = parse_agent.run(f"用户回答：{ans}\n当前事实库：{json.dumps(facts, ensure_ascii=False)}"
                               "\n请输出 updates JSON")
        updates = _extract_json(resp).get("updates", {})
        for k, v in updates.items():
            if k in facts:
                facts[k] = v
        save_facts(facts)
        print("已更新事实库 ✓")
    export_all(facts)
    print("三语简历已重新导出到 outputs/")

## 第5部分：性能评估

1. 和暦换算正确性（含全部元号切换边界）
2. 润色后的数字一致性（防止 LLM 篡改/遗漏事实）
3. 各环节耗时统计

In [ ]:
# ==================================================
# 第5部分：性能评估
# ==================================================

print("=== 评估 1：和暦换算正确性（元号边界） ===")
cases = [
    ("2023-09", "令和5年9月"),
    ("2019-04", "平成31年4月"),
    ("2019-05", "令和元年5月"),
    ("1989-01", "昭和64年1月"),
    ("1989-01-08", "平成元年1月8日"),
    ("1989-02", "平成元年2月"),
    ("1926-12", "大正15年12月"),
    ("1926-12-25", "昭和元年12月25日"),
    ("1927-01", "昭和2年1月"),
    ("1912-07", "明治45年7月"),
    ("1912-08", "大正元年8月"),
]
passed = 0
for iso, want in cases:
    got = to_wareki_date(iso)
    flag = got == want
    passed += flag
    print(("✓" if flag else "✗"), iso, "→", got
          + ("" if flag else f"（应为 {want}）"))
print(f"和暦换算通过 {passed}/{len(cases)}")
assert passed == len(cases)

assert from_wareki("令和5年9月")[:2] == (2023, 9)
print("✓ 和暦→西暦 往返换算通过")

print("\n=== 评估 2：润色数字一致性（防篡改） ===")
raw_bullets = [b for x in load_facts()["experiences"] for b in x.get("bullets", [])]
if POLISHED_PATH.exists():
    pol = json.loads(POLISHED_PATH.read_text(encoding="utf-8"))

    # 单位归一化：500万 与 5 million 视为同一数值，避免翻译误报
    UNITS = {"万": 1e4, "亿": 1e8, "千": 1e3,
             "million": 1e6, "billion": 1e9, "thousand": 1e3, "k": 1e3, "m": 1e6}
    NUM_PAT = re.compile(r"(\d+(?:\.\d+)?)\s*(万|亿|千|million|billion|thousand|k|m)\b", re.I)

    def _values(s):
        vals = []
        for m in NUM_PAT.finditer(s):
            unit = (m.group(2) or "").lower()
            vals.append(float(m.group(1)) * UNITS.get(unit, 1))
        return sorted(round(v, 6) for v in vals)

    for lang, key in [("zh", "bullets_zh"), ("en", "bullets_en"), ("ja", "bullets_ja")]:
        orig, new = _values(" ".join(raw_bullets)), _values(" ".join(pol.get(key, [])))
        missing = [v for v in orig if v not in new]
        print(("✓" if not missing else "✗"),
              f"[{lang}] 关键数字保留完整" + ("" if not missing else f"，缺失: {missing}"))
        assert not missing
    print("✓ 三语润色均未篡改/遗漏关键数字（500万 ≡ 5 million 单位归一）")
else:
    print("（未运行润色演示，跳过一致性检查）")

print("\n=== 评估 3：生成耗时 ===")
t0 = time.perf_counter()
export_all(load_facts(), polished if POLISHED_PATH.exists() else None)
dt_local = time.perf_counter() - t0
print(f"本地渲染四份文档：{dt_local:.2f}s（不含 LLM 调用）")
print("\n评估完成 ✓")

## 第6部分：总结与展望

### 项目总结

#### 实现的功能
- [x] Agent 访谈式信息收集（提问 + 回答解析入库）
- [x] 结构化事实库（唯一数据源，落盘可恢复）
- [x] 和暦换算工具（含元号边界处理，可被 Agent 调用）
- [x] 三语独立生成：中文单页 / 日文履歴書＋職務経歴書 / 英文 ATS 版式
- [x] 三语润色与摘要（数字一致性校验）
- [x] 模拟 HR / ATS 评审
- [x] Markdown / HTML / PDF 导出

#### 遇到的挑战
- 和暦元号切换边界（1989/2019 等跨年）：以月份为粒度定义边界规则，并用 9 组用例验证
- LLM 润色可能篡改事实数字：增加输出后数字集合比对校验
- PDF 中文乱码：改为浏览器内核打印方案，复用系统 CJK 字体

#### 未来改进方向
- [ ] 粘贴目标 JD，自动标注简历缺失关键词
- [ ] 多版本管理（同一事实库派生多个投递版本）
- [ ] 多模板切换与 B5 纸张支持
- [ ] 上传已有简历（PDF/DOCX）解析导入

### 参考
- Hello-Agents 框架：https://github.com/datawhalechina/hello-agents